# STARDAR — Phase 1: Bistatic Geometry Exploration

This notebook explores the bistatic geometry of a Starlink satellite pass over a fixed ground receiver.

**Goals:**
- Visualize satellite ground track and bistatic geometry
- Compute range, Doppler, and bistatic angle over a full pass
- Estimate achievable range and azimuth resolution
- Run a first-pass SNR budget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from simulation.geometry.bistatic import (
    BistaticGeometry, bistatic_range, bistatic_angle,
    bistatic_doppler, range_resolution, azimuth_resolution
)
from simulation.orbital.starlink import (
    STARLINK_WAVELENGTH_M, STARLINK_BANDWIDTH_HZ,
    STARLINK_VELOCITY_MS, pass_duration_seconds
)

print(f'Wavelength: {STARLINK_WAVELENGTH_M*100:.1f} cm')
print(f'Max bandwidth: {STARLINK_BANDWIDTH_HZ/1e6:.0f} MHz')
print(f'Range resolution: {range_resolution(STARLINK_BANDWIDTH_HZ):.2f} m')
print(f'Approx pass duration (>10 deg): {pass_duration_seconds(10):.0f} s')

## 1. Simplified Geometry (Local ENU coordinates)

For initial analysis we work in a local East-North-Up frame centered on the receive array.
Satellite flies overhead at 550 km altitude.

In [ ]:
# --- Scene setup ---
SAT_ALT   = 550e3        # m
SAT_VEL   = 7600.0       # m/s (along-track)
FREQ      = 12e9         # Hz
LAMBDA    = 3e8 / FREQ   # m
BW        = 250e6        # Hz

# Receiver at origin
rx_pos = np.array([0.0, 0.0, 0.0])

# Target 20 km north, on the ground
target_pos = np.array([0.0, 20e3, 0.0])

# Satellite passes from south to north, 50 km west offset (not directly overhead)
# Time vector: -250s to +250s around closest approach
t = np.linspace(-250, 250, 1000)   # seconds
sat_x = -50e3 * np.ones_like(t)   # 50 km west
sat_y = SAT_VEL * t               # moving north
sat_z = SAT_ALT * np.ones_like(t)

sat_vel = np.array([0.0, SAT_VEL, 0.0])

# --- Compute geometry over pass ---
RT_arr, RR_arr, R_bi_arr, beta_arr, fd_arr = [], [], [], [], []

for i in range(len(t)):
    tx_pos = np.array([sat_x[i], sat_y[i], sat_z[i]])
    geom = BistaticGeometry(tx_pos, sat_vel, rx_pos, target_pos)
    RT, RR, Rbi = bistatic_range(geom)
    beta = bistatic_angle(geom)
    fd = bistatic_doppler(geom, LAMBDA)
    RT_arr.append(RT); RR_arr.append(RR)
    R_bi_arr.append(Rbi); beta_arr.append(np.degrees(beta))
    fd_arr.append(fd)

RT_arr = np.array(RT_arr)
RR_arr = np.array(RR_arr)
R_bi_arr = np.array(R_bi_arr)
beta_arr = np.array(beta_arr)
fd_arr = np.array(fd_arr)

print('Geometry computed over', len(t), 'time steps')
print(f'RT range: {RT_arr.min()/1e3:.1f} – {RT_arr.max()/1e3:.1f} km')
print(f'RR (fixed): {RR_arr[0]/1e3:.1f} km')
print(f'Bistatic angle: {beta_arr.min():.1f}° – {beta_arr.max():.1f}°')
print(f'Max Doppler: {fd_arr.max():.0f} Hz')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Ground track + geometry
ax = axes[0, 0]
ax.plot(sat_x/1e3, sat_y/1e3, 'b-', linewidth=2, label='Satellite ground track')
ax.plot(sat_x[0]/1e3, sat_y[0]/1e3, 'b>', markersize=10)
ax.plot(rx_pos[0]/1e3, rx_pos[1]/1e3, 'g^', markersize=14, label='Rx array', zorder=5)
ax.plot(target_pos[0]/1e3, target_pos[1]/1e3, 'rx', markersize=14, markeredgewidth=3, label='Target', zorder=5)
ax.set_xlabel('East (km)'); ax.set_ylabel('North (km)')
ax.set_title('Ground Projection'); ax.legend(); ax.grid(alpha=0.3)
ax.set_aspect('equal')

# Bistatic range
ax = axes[0, 1]
ax.plot(t, RT_arr/1e3, 'b-', label='RT (sat→target)')
ax.plot(t, RR_arr/1e3, 'g--', label='RR (target→rx)')
ax.plot(t, R_bi_arr/1e3, 'r-', linewidth=2, label='R_bistatic = RT+RR')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Range (km)')
ax.set_title('Bistatic Range vs. Time'); ax.legend(); ax.grid(alpha=0.3)

# Bistatic angle
ax = axes[1, 0]
ax.plot(t, beta_arr, 'purple', linewidth=2)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Bistatic angle β (deg)')
ax.set_title('Bistatic Angle vs. Time'); ax.grid(alpha=0.3)

# Doppler
ax = axes[1, 1]
ax.plot(t, fd_arr/1e3, 'r-', linewidth=2)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Doppler (kHz)')
ax.set_title('Bistatic Doppler vs. Time'); ax.grid(alpha=0.3)

plt.suptitle('STARDAR — Bistatic Geometry Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Resolution Analysis

In [ ]:
# Range resolution vs bandwidth
bw_values = np.logspace(6, 9, 200)   # 1 MHz to 1 GHz
range_res = 3e8 / (2 * bw_values)

# Azimuth resolution vs integration time
T_int_values = np.linspace(1, 300, 200)  # 1s to 5 min
slant_range = 560e3   # approx RT at moderate elevation
az_res = (LAMBDA * slant_range) / (2 * SAT_VEL * T_int_values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.loglog(bw_values/1e6, range_res, 'b-', linewidth=2)
ax.axvline(250, color='r', linestyle='--', label='Starlink max BW (250 MHz)')
ax.axhline(range_resolution(250e6), color='r', linestyle=':')
ax.set_xlabel('Bandwidth (MHz)'); ax.set_ylabel('Range resolution (m)')
ax.set_title('Range Resolution vs. Bandwidth')
ax.legend(); ax.grid(True, which='both', alpha=0.3)

ax = axes[1]
ax.plot(T_int_values, az_res, 'g-', linewidth=2)
ax.axhline(1.0, color='r', linestyle='--', label='1 m resolution')
ax.set_xlabel('Integration time (s)'); ax.set_ylabel('Azimuth resolution (m)')
ax.set_title(f'Azimuth Resolution vs. Integration Time\n(R={slant_range/1e3:.0f} km, V={SAT_VEL} m/s, λ={LAMBDA*100:.1f} cm)')
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('STARDAR — Resolution Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Range resolution @ 250 MHz BW: {range_resolution(250e6):.2f} m')
t_for_1m = (LAMBDA * slant_range) / (2 * SAT_VEL * 1.0)
print(f'Integration time needed for 1m azimuth res: {t_for_1m:.1f} s')

## 3. SNR Budget

In [ ]:
# Bistatic radar equation — all in dB
c    = 3e8
k_B  = 1.38e-23  # Boltzmann constant

# Parameters
EIRP_dBW     = 37.0    # Starlink spot beam EIRP estimate
RT_m         = 560e3   # Tx-to-target range
RR_m         = 20e3    # Target-to-Rx range
wavelength   = LAMBDA
sigma_dBsm   = 10.0    # target RCS — aircraft ~10 m²
Gr_dBi       = 30.0    # receive array gain (placeholder)
T_sys_K      = 300.0   # system noise temperature
B_hz         = 250e6   # receiver bandwidth
T_int_s      = 10.0    # integration time
L_dB         = 3.0     # misc losses

# Compute each term
lambda_term_dB = 20 * np.log10(wavelength)
RT_loss_dB     = -20 * np.log10(4 * np.pi) - 20 * np.log10(RT_m)
RR_loss_dB     = -20 * np.log10(4 * np.pi) - 20 * np.log10(RR_m)
noise_dB       = 10 * np.log10(k_B * T_sys_K * B_hz)
int_gain_dB    = 10 * np.log10(B_hz * T_int_s)  # matched filter + integration

SNR_dB = (EIRP_dBW + Gr_dBi + 2 * lambda_term_dB
          + sigma_dBsm + RT_loss_dB + RR_loss_dB
          - noise_dB + int_gain_dB - L_dB)

budget = {
    'EIRP':          EIRP_dBW,
    'Gr (array)':    Gr_dBi,
    'λ² term':       2 * lambda_term_dB,
    'RCS (σ)':       sigma_dBsm,
    'RT path loss':  RT_loss_dB,
    'RR path loss':  RR_loss_dB,
    'Noise floor':   -noise_dB,
    'Integ. gain':   int_gain_dB,
    'Losses':        -L_dB,
}

from simulation.visualization.plots import plot_snr_budget
fig, ax = plot_snr_budget(budget, title=f'SNR Budget — STARDAR (total: {SNR_dB:.1f} dB)')
plt.show()

print(f'\nEstimated SNR: {SNR_dB:.1f} dB')
print(f'(Integration time: {T_int_s}s, target RCS: {sigma_dBsm} dBsm, Gr: {Gr_dBi} dBi)')